In [1]:
import lmstudio as lms
import pandas as pd
import os
from PIL import Image
from sklearn.metrics import *
from sklearn.preprocessing import LabelEncoder
import time
from colorama import Fore, Style
import re
import gc

In [2]:
os.environ["CUDA_VISIBLE_DEVICES"] = "1" # Only use the 2nd GPU

In [3]:
# Qwen
#MODEL_NAME = "qwen/qwen2.5-vl-7b"
#MODEL_NAME = "qwen/qwen3-vl-4b"
#MODEL_NAME = "qwen/qwen3-vl-8b"
#MODEL_NAME = "qwen/qwen3-vl-30b"

# Gemma
#MODEL_NAME = "google/gemma-3-4b"
#MODEL_NAME = "google/gemma-3-12b"
#MODEL_NAME = "google/gemma-3-27b"

#MODEL_NAME = "google/gemma-4-26b-a4b"
#MODEL_NAME = "gemma-4-e2b-it"
MODEL_NAME = "gemma-4-e4b-it"

# LMF2 LiquidAI
#MODEL_NAME = "liquidai_lfm2-vl-450m"
#MODEL_NAME = "lfm2-vl-1.6b"
#MODEL_NAME = "lfm2-vl-3b"


# MistralAI
#MODEL_NAME = "mistralai/ministral-3-3b"
#MODEL_NAME = "mistralai/magistral-small-2509" # This is a reasoning model, which is expected to be slower

model = lms.llm(MODEL_NAME)

In [4]:
# Example

sample_img = "1JQk5NF.png" # Actual label is 1

In [ ]:
# Setup file paths
IMG_DIR   = "/home/sw_nsolagratia/Documents/Multimodal Project/MultiOFF_Dataset/Labelled Images/"
test_dataset = "/home/sw_nsolagratia/Documents/Multimodal Project/MultiOFF_Dataset/Split Dataset/Testing_meme_dataset.csv"


In [6]:
def strip_thinking(text: str) -> str:
    # Clean standard <thought> or [think] blocks
    text = re.sub(
        r"[\[<]\s*(?:think|thought)\s*[\]>].*?[\[<]\s*/\s*(?:think|thought)\s*[\]>]",
        "",
        text,
        flags=re.IGNORECASE | re.DOTALL,
    )
    
    # Remove <channel> blocks
    text = re.sub(
        r"<\|channel>.*?<channel\|>", 
        "", 
        text, 
        flags=re.DOTALL | re.IGNORECASE
    )
    
    return text.strip()

In [7]:
# Warm up

result = model.respond("Hi how are you")
print(result)

I'm doing well, thank you for asking! 😊

How are you today? Is there anything I can help you with or anything you'd like to chat about?


In [8]:
result = str(result)
print(strip_thinking(result))

I'm doing well, thank you for asking! 😊

How are you today? Is there anything I can help you with or anything you'd like to chat about?


In [9]:
example_path = os.path.join(IMG_DIR, sample_img)
example_img_handle = lms.prepare_image(example_path)

In [10]:
# Warming up with image input
# Actual / Correct label is 1

for i in range(10):
    chat = lms.Chat()

    chat.add_user_message(
        "Reply with ONLY 1 for Yes OR 0 for NO. Nothing else. Do not describe the image. The question being: is this image considered hate speech or offensive?",
        images=[example_img_handle]
    )
    example_response = str(model.respond(chat))
    example_response = strip_thinking(example_response) 
    print(f"Response no. {i+1}\t: {example_response}")

    del chat

Response no. 1	: 0
Response no. 2	: 0
Response no. 3	: 0
Response no. 4	: 0
Response no. 5	: 0
Response no. 6	: 0
Response no. 7	: 0
Response no. 8	: 0
Response no. 9	: 0
Response no. 10	: 0


In [11]:
#model.unload() 

In [12]:
# Setup dataset
#df = pd.read_csv(dataset) # Entire dataset
df = pd.read_csv(test_dataset) # Test dataset
display(df) # before
# Label encoding
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['label'].tolist())
filenames = df['image_name'].tolist()
actual_labels = df['label'].tolist()

,image_name,sentence,label
0,jyxHhiB.png,3 hrs Black nurse in Connecticut asked me if T...,offensive
1,we4hhWi.png,"I do n't believe that women have any rights , ...",offensive
2,TpIZoZr.png,WHY THE FUCK DOES IRITHE DONALD HAVE BETTER NE...,offensive
3,h6Pkqkr.png,"After the Bern subsides , get ready to ... Fee...",Non-offensiv
4,94anjQG.png,"MAG & WALLY Mrs. Clinton , I 'm awesome ! cong...",Non-offensiv
...,...,...,...
143,fyAh3I0.png,WHAT IF POKEMON GO WAS RELEASED TO D US FRO TH...,Non-offensiv
144,CMQqhImUkAAKeno.jpg,Donald Trump 's hair looks like someone tried ...,Non-offensiv
145,tINjUCc.png,MAMAS Who am I supposed to vote for ? Am l sup...,Non-offensiv
146,eI2N5iQ.png,EcakpnBeHn rnacoBnl 18 minutes ago We will hav...,Non-offensiv


In [13]:
display(df) # after

,image_name,sentence,label
0,jyxHhiB.png,3 hrs Black nurse in Connecticut asked me if T...,1
1,we4hhWi.png,"I do n't believe that women have any rights , ...",1
2,TpIZoZr.png,WHY THE FUCK DOES IRITHE DONALD HAVE BETTER NE...,1
3,h6Pkqkr.png,"After the Bern subsides , get ready to ... Fee...",0
4,94anjQG.png,"MAG & WALLY Mrs. Clinton , I 'm awesome ! cong...",0
...,...,...,...
143,fyAh3I0.png,WHAT IF POKEMON GO WAS RELEASED TO D US FRO TH...,0
144,CMQqhImUkAAKeno.jpg,Donald Trump 's hair looks like someone tried ...,0
145,tINjUCc.png,MAMAS Who am I supposed to vote for ? Am l sup...,0
146,eI2N5iQ.png,EcakpnBeHn rnacoBnl 18 minutes ago We will hav...,0


In [14]:
# List to store predictions
predictions = []

In [15]:
# Set the batch size to fit the token limit
batch_size = 10

# Split the filenames into batches
batches = [filenames[i:i + batch_size] for i in range(0, len(filenames), batch_size)]

# Track processed images to avoid repeats
processed_images = set()

label_map = dict(zip(df['image_name'], df['label'])) # Map filenames to actual labels

# Store results
results_list = []

In [16]:
from tqdm.auto import tqdm 

# Calculate total images for a more accurate progress bar if possible
total_batches = len(batches)

# Start Timer
start_time = time.perf_counter()

# Wrap the outer loop with tqdm
for batch in tqdm(batches, desc="Processing Batches", unit="batch"):
    try:
        for filename in batch:
            # Reset chat for each image to ensure a clean start
            chat = lms.Chat()
            
            if filename in processed_images:
                continue

            image_path = os.path.join(IMG_DIR, filename)
            image_handle = lms.prepare_image(image_path)

            processed_images.add(filename)

            chat.add_user_message(
                "Reply with ONLY 1 for Yes OR 0 for NO. Nothing else. Do not describe the image. The question being: is this image considered hate speech or offensive?",
                images=[image_handle]
            )

            # Get the response
            response = str(model.respond(chat))
            response = strip_thinking(response)

            # Normalize and handle response
            single_prediction = -1  
            response_str = str(response).strip()
            
            if isinstance(response, list):
                if response and str(response[0]).strip() in ["0", "1"]:
                    single_prediction = int(str(response[0]).strip())
            elif response_str in ["0", "1"]:
                single_prediction = int(response_str)

            # Collect and store result
            predictions.append(single_prediction)
            actual_label = label_map.get(filename, 'N/A')

            results_list.append({
                'filename': filename,
                'actual_label': actual_label,
                'predicted_label': single_prediction
            })

            del image_handle 
            gc.collect()

    except Exception as e:
        print(f"\n{Style.BRIGHT}{Fore.RED}Error occurred during batch processing: {e}{Style.RESET_ALL}")
        model.unload()
        break

# End Timer
end_time = time.perf_counter()
elapsed_time = end_time - start_time

# Create the final DataFrame from the list of results
results_df = pd.DataFrame(results_list)

# Display the head of the results DataFrame
print("\n--- Results DataFrame Head ---")
display(results_df.head())
print("----------------------------\n")

# Print Elapsed Time 
hours, remainder = divmod(elapsed_time, 3600)
minutes, seconds = divmod(remainder, 60)
print(f"{Style.BRIGHT}{Fore.MAGENTA}Total Evaluation Time: {int(hours):02d}h {int(minutes):02d}m {seconds:05.2f}s{Style.RESET_ALL}")

# Final accuracy calculation
# Filter out samples where prediction failed (-1) before calculating accuracy
valid_results_df = results_df[results_df['predicted_label'].isin([0, 1])].copy()
valid_predictions = valid_results_df['predicted_label'].tolist()
valid_actuals = valid_results_df['actual_label'].tolist()

Processing Batches:   0%|          | 0/15 [00:00<?, ?batch/s]


--- Results DataFrame Head ---


,filename,actual_label,predicted_label
0,jyxHhiB.png,1,0
1,we4hhWi.png,1,0
2,TpIZoZr.png,1,0
3,h6Pkqkr.png,0,0
4,94anjQG.png,0,0


----------------------------

Total Evaluation Time: 00h 01m 13.50s


In [17]:
print(f"{Style.BRIGHT}{Fore.BLUE}Total processed images: {len(results_df)}{Style.RESET_ALL}")
print(f"{Style.BRIGHT}{Fore.GREEN}Valid predictions used for metrics: {len(valid_predictions)}{Style.RESET_ALL}")
time_str = f"{int(hours):02d}:{int(minutes):02d}:{int(seconds):02d}" 
print(f"{Style.BRIGHT}{Fore.MAGENTA}Total Evaluation Time: {time_str}{Style.RESET_ALL}")


if len(valid_predictions) > 0:
    # Prepare the list for corrected predictions, treating -1 as incorrect
    results_df['predicted_label_corrected'] = results_df['predicted_label'].apply(lambda x: x if x != -1 else None)

    # Filter out rows where predicted_label is None (i.e., -1 values)
    valid_results_df = results_df[results_df['predicted_label_corrected'].notna()]

    valid_predictions = valid_results_df['predicted_label_corrected'].tolist()
    valid_actuals = valid_results_df['actual_label'].tolist()

    # Calculate metrics, treating -1 as incorrect but keeping 0 as valid if it matches the actual label
    accuracy = accuracy_score(valid_actuals, valid_predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        valid_actuals, valid_predictions, labels=None, pos_label=1, average=None, zero_division=0, beta=1.0
    )
    mcc = matthews_corrcoef(valid_actuals, valid_predictions)

    # Calculate correct predictions
    correct_predictions = sum(1 for p, a in zip(valid_predictions, valid_actuals) if p == a)

    # Print the metrics
    print(f"\n--- {Style.BRIGHT}{Fore.YELLOW}{MODEL_NAME} Performance Metrics ---{Style.RESET_ALL}")
    print(f"{Style.BRIGHT}{Fore.GREEN}Accuracy\t: {accuracy * 100:.4f}%{Style.RESET_ALL}")
    print(f"{Style.BRIGHT}{Fore.GREEN}Precision\t: {', '.join(f'{p * 100:.4f}%' for p in precision)}{Style.RESET_ALL} ")
    print(f"{Style.BRIGHT}{Fore.GREEN}Recall\t\t: {', '.join(f'{r * 100:.4f}%' for r in recall)}{Style.RESET_ALL} ")
    print(f"{Style.BRIGHT}{Fore.GREEN}F1 Score\t: {', '.join(f'{f1_score * 100:.4f}%' for f1_score in f1)}{Style.RESET_ALL} ")
    print(f"{Style.BRIGHT}{Fore.GREEN}MCC\t\t: {mcc * 100:.4f}%{Style.RESET_ALL}")
    print(f"{Style.BRIGHT}{Fore.YELLOW}{'-'*50}\n{Style.RESET_ALL}")
    print(f"{Style.BRIGHT}{Fore.GREEN}Out of {len(valid_predictions)} valid predictions, correctly predicted {correct_predictions} labels{Style.RESET_ALL}")
    print(f"{Style.BRIGHT}{Fore.CYAN}\nClassification Report:\n\n{classification_report(valid_actuals, valid_predictions, digits=4)}{Style.RESET_ALL}")
else:
    print(f"{Style.BRIGHT}{Fore.RED}No valid predictions (0 or 1) were collected for metrics calculation.{Style.RESET_ALL}")

Total processed images: 148
Valid predictions used for metrics: 148
Total Evaluation Time: 00:01:13

--- gemma-4-e4b-it Performance Metrics ---
Accuracy	: 64.1892%
Precision	: 63.3803%, 83.3333% 
Recall		: 98.9011%, 8.7719% 
F1 Score	: 77.2532%, 15.8730% 
MCC		: 18.9324%
--------------------------------------------------

Out of 148 valid predictions, correctly predicted 95 labels

Classification Report:

              precision    recall  f1-score   support

           0     0.6338    0.9890    0.7725        91
           1     0.8333    0.0877    0.1587        57

    accuracy                         0.6419       148
   macro avg     0.7336    0.5384    0.4656       148
weighted avg     0.7106    0.6419    0.5361       148



In [18]:
model.unload() # Eject the model from memory
print("Model unloaded")

Model unloaded
